<div align="center">

# 🧠 Semana 1 · Tu primer modelo de IA

**Máster en Creación de Modelos de IA · Nivel I: «Construye Gleihar»** · SagaTechSchool Pro

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saga1555/SagaEstudios7316/blob/main/semana01/semana01_tu_primer_modelo.ipynb)

</div>

---

En la teoría aprendiste que **un modelo es una función ŷ = f(x; θ)** y que **aprender es ajustar los parámetros θ para que la pérdida sea mínima**. En este cuaderno lo vas a comprobar con tus propias manos.

### 🎯 Al terminar podrás

1. Construir un modelo lineal con sus **cuatro componentes**: datos, modelo, pérdida y optimización.
2. Ver cómo **baja el error** cuando los parámetros mejoran.
3. Usar el modelo para **predecir** (inferencia) y **evaluarlo** con un dato nuevo.
4. Reconocer los mismos cuatro componentes en **Gleihar v0.0**, tu primer modelo de lenguaje.

### 🗺️ Mapa del cuaderno

| Parte | Qué haces | Capítulo de la teoría | Tiempo |
|---|---|---|---|
| 0 · Tu entorno | Compruebas qué recursos tienes | Cap. 7 | 10 min |
| 1 · Datos | Miras los datos de los terrenos | Cap. 2 | 10 min |
| 2 · Modelo y pérdida | Programas el modelo y su error | Caps. 1 y 2 | 30 min |
| 3 · Optimización | Buscas los mejores parámetros | Cap. 2 | 30 min |
| 4 · Inferencia y evaluación | Predices y evalúas | Caps. 2 y 3 | 15 min |
| 5 · Gleihar v0.0 | Un modelo de lenguaje con los mismos 4 componentes | Cap. 6 | 40 min |
| 6 · Entrega | Generas tu reporte | — | 10 min |

### 🧭 Cómo leer este cuaderno

| Símbolo | Significa |
|---|---|
| 🔮 **Predice** | Antes de ejecutar, escribe qué crees que pasará. |
| ✍️ **Tu turno** | Completa el código donde dice `...`. |
| ✅ **Comprueba** | Pregunta rápida; haz clic para ver la respuesta. |
| ⚠️ **Cuidado** | Error frecuente y cómo evitarlo. |

> 💡 **Antes de empezar:** guarda tu copia con *Archivo → Guardar una copia en Drive*. Activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU* (esta semana todo funciona también sin GPU). Ejecuta las celdas **en orden** con `Shift + Enter`.

## 0️⃣ Tu entorno: ¿con qué recursos cuentas?

En el capítulo 7 viste que crear un modelo requiere **datos, cómputo y herramientas**. Esta celda revisa tu cómputo: si tienes GPU, de qué generación es y cuánta memoria tiene. Con eso elige la configuración que usaremos en las próximas semanas. No tienes que modificar nada.

In [ ]:
# ══════════════ GLEIHAR-INICIO · revisa tu entorno ══════════════
import sys, platform, json, time
import torch

EN_COLAB = "google.colab" in sys.modules
GLEIHAR = {"entorno": "Google Colab" if EN_COLAB else "Equipo local",
           "python": platform.python_version(), "pytorch": torch.__version__,
           "dispositivo": "cpu", "gpu": None, "vram_gb": 0.0, "capacidad": None,
           "precision": "float32", "avisos": []}

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    mayor, menor = torch.cuda.get_device_capability(0)
    GLEIHAR.update(dispositivo="cuda", gpu=p.name, vram_gb=round(p.total_memory / 1024**3, 1),
                   capacidad=f"{mayor}.{menor}")
    if f"sm_{mayor}{menor}" not in torch.cuda.get_arch_list():
        GLEIHAR["avisos"].append("Tu PyTorch no trae soporte para esta tarjeta. Revisa el anexo técnico del curso.")
    GLEIHAR["precision"] = "bfloat16" if mayor >= 8 else ("float16" if mayor == 7 else "float32")
else:
    GLEIHAR["avisos"].append("Sin GPU: esta semana todo funciona igual en CPU.")

DISPOSITIVO = torch.device(GLEIHAR["dispositivo"])
print("═" * 52); print("  GLEIHAR-INICIO · tu entorno"); print("═" * 52)
for k in ["entorno", "python", "pytorch", "dispositivo", "gpu", "vram_gb", "capacidad", "precision"]:
    print(f"  {k:<12}: {GLEIHAR[k]}")
for a in GLEIHAR["avisos"]:
    print("  ⚠", a)

> ✅ **Comprueba:** ¿tu informe dice `cuda` o `cpu`? Si estás en Colab y dice `cpu`, no activaste la GPU. No es grave esta semana, pero actívala para acostumbrarte.

## 1️⃣ Componente 1 · Los datos

Usamos los cinco terrenos de la teoría (capítulo 2). Las superficies están en **cientos de m²** y los precios en **miles de dólares**, para trabajar con números pequeños.

In [ ]:
import matplotlib.pyplot as plt

superficie = [1.0, 1.5, 2.0, 2.5, 3.0]     # x: cientos de m²
precio     = [11.5, 18.0, 22.5, 28.5, 33.0] # y: miles de dólares

plt.figure(figsize=(6, 4))
plt.scatter(superficie, precio, s=70, color="#6C5CE7")
for xi, yi, n in zip(superficie, precio, "ABCDE"):
    plt.annotate(n, (xi, yi), textcoords="offset points", xytext=(6, -12))
plt.xlabel("Superficie (cientos de m²)"); plt.ylabel("Precio (miles de $)")
plt.title("Terrenos de un sector de Milagro (datos ilustrativos)")
plt.grid(alpha=.3); plt.show()

> 🔮 **Predice:** mirando el gráfico, ¿cuánto crees que costaría un terreno de 220 m² (x = 2,2)? Anota tu estimación: al final la compararás con la del modelo.

## 2️⃣ Componentes 2 y 3 · El modelo y la función de pérdida

**El modelo** (capítulo 1) es una recta con dos parámetros, θ = (w, b):

$$\hat{y} = w \cdot x + b$$

**La pérdida** (capítulo 2) es el error cuadrático medio:

$$\text{ECM} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### ✍️ Tu turno

Completa las dos funciones reemplazando `...`. Cada una lleva un ejemplo en el comentario para que compruebes tu resultado.

In [ ]:
def predecir(x, w, b):
    """Devuelve la predicción del modelo lineal.
    Ejemplo: predecir(2.0, 10, 1) → 21.0"""
    ...

def ecm(w, b):
    """Error cuadrático medio del modelo (w, b) sobre los 5 terrenos.
    Pista: recorre zip(superficie, precio), acumula (y - predecir(x, w, b))**2 y divide entre len(precio).
    Ejemplo: ecm(10, 1) → 3.35"""
    ...

In [ ]:
# ── Revisión automática (no la modifiques) ──
def revisar_parte2():
    pruebas = [("predecir", lambda: abs(predecir(2.0, 10, 1) - 21.0) < 1e-9),
               ("ecm", lambda: abs(ecm(10, 1) - 3.35) < 1e-9 and abs(ecm(11, 0.5) - 0.30) < 1e-9)]
    ok_total = 0
    for nombre, prueba in pruebas:
        try: ok = bool(prueba())
        except Exception: ok = False
        ok_total += ok
        print(("✅ " if ok else "❌ ") + nombre)
    GLEIHAR["parte2"] = f"{ok_total}/{len(pruebas)}"
    print(f"Resultado: {ok_total}/{len(pruebas)}")
revisar_parte2()

<details><summary><b>🆘 ¿Atascado? Pistas</b></summary>

- `predecir`: `return w * x + b`
- `ecm`:
```python
total = 0
for x, y in zip(superficie, precio):
    total += (y - predecir(x, w, b)) ** 2
return total / len(precio)
```
</details>

> ⚠️ **Cuidado:** si la revisión sale ❌, verifica que tu función use `return`. Una función sin `return` devuelve `None`.

In [ ]:
# Los tres intentos del capítulo 2
assert ecm(10, 1) is not None, "Primero completa predecir y ecm (deben usar return)."
for w, b in [(10, 1), (11, 0.5), (10.7, 1.3)]:
    print(f"w = {w:<5} b = {b:<4} →  ECM = {ecm(w, b):.3f}")

> ✅ **Comprueba:** ¿obtuviste 3,35 · 0,30 · 0,215, como en el ejemplo resuelto de la teoría? Si sí, tu modelo y tu pérdida están bien programados.

## 3️⃣ Componente 4 · La optimización: buscar los mejores parámetros

Aprender es encontrar los (w, b) con la pérdida más pequeña. Empezamos con la estrategia más sencilla: **probar muchísimas combinaciones** y quedarnos con la mejor. Se llama *búsqueda en cuadrícula*.

> 🔮 **Predice:** ¿cerca de qué valores de w y b crees que estará el mínimo?

In [ ]:
mejor = (None, None, float("inf"))
pruebas = 0
for i in range(81):                 # w de 8.0 a 13.0 en pasos de 0.0625
    w = 8 + i * 0.0625
    for j in range(81):             # b de -2.0 a 3.0 en pasos de 0.0625
        b = -2 + j * 0.0625
        e = ecm(w, b); pruebas += 1
        if e < mejor[2]:
            mejor = (w, b, e)
w_mejor, b_mejor, e_mejor = mejor
print(f"Probé {pruebas} combinaciones.")
print(f"Mejores parámetros: w = {w_mejor:.3f}, b = {b_mejor:.3f}  →  ECM = {e_mejor:.4f}")

Para una recta existe una fórmula exacta, **mínimos cuadrados**. NumPy la trae lista en `np.polyfit`. Comparemos:

In [ ]:
import numpy as np
w_exacto, b_exacto = np.polyfit(superficie, precio, 1)
print(f"Mínimos cuadrados: w = {w_exacto:.3f}, b = {b_exacto:.3f}  →  ECM = {ecm(w_exacto, b_exacto):.4f}")

# Visualiza cómo cambia la pérdida al mover w (con b fijo en el óptimo)
ws = [8 + k * 0.05 for k in range(101)]
plt.figure(figsize=(6, 3.5))
plt.plot(ws, [ecm(w, b_exacto) for w in ws], color="#6C5CE7")
plt.axvline(w_exacto, ls="--", color="#999"); plt.xlabel("w"); plt.ylabel("ECM")
plt.title("La pérdida tiene forma de valle: aprender es bajar al fondo"); plt.grid(alpha=.3); plt.show()

> ✅ **Comprueba:** ¿por qué la curva tiene forma de valle?
> <details><summary>Ver respuesta</summary>Porque el ECM suma errores al cuadrado: si w es demasiado pequeño o demasiado grande, el error crece. El fondo del valle es el mejor w. En la semana 3 aprenderás a «bajar» por ese valle con el <b>descenso de gradiente</b>, el método que se usa cuando hay millones de parámetros y probar todas las combinaciones es imposible.</details>

## 4️⃣ Inferencia y evaluación

El modelo ya está **entrenado**. Ahora lo usamos para **predecir** un caso nuevo (inferencia) y lo **evaluamos** con un terreno que **no** usamos para entrenar.

In [ ]:
def dolares(miles):            # formato ecuatoriano: $24 840
    return "$" + f"{miles * 1000:,.0f}".replace(",", " ")

x_nuevo = 2.2
print(f"Predicción para 220 m²: {dolares(predecir(x_nuevo, w_exacto, b_exacto))}")

# Evaluación con un terreno que el modelo nunca vio: F = 180 m² vendido en $20 000
x_eval, y_eval = 1.8, 20.0
y_hat = predecir(x_eval, w_exacto, b_exacto)
print(f"Terreno F · real: {dolares(y_eval)} · predicho: {dolares(y_hat)} · error: {dolares(abs(y_eval - y_hat))}")

plt.figure(figsize=(6, 4))
plt.scatter(superficie, precio, s=70, color="#6C5CE7", label="Entrenamiento")
plt.scatter([x_eval], [y_eval], s=90, marker="s", color="#E17055", label="Evaluación (F)")
xs = [0.8, 3.2]; plt.plot(xs, [predecir(x, w_exacto, b_exacto) for x in xs], color="#1B2140", label="Modelo")
plt.xlabel("Superficie (cientos de m²)"); plt.ylabel("Precio (miles de $)"); plt.legend(); plt.grid(alpha=.3); plt.show()
GLEIHAR["modelo_terrenos"] = {"w": round(float(w_exacto), 3), "b": round(float(b_exacto), 3),
                              "ecm": round(float(ecm(w_exacto, b_exacto)), 4)}

> ✅ **Comprueba:** ¿el error con el terreno F es menor que $1 000, el criterio de éxito que definimos en el capítulo 3? ¿Cómo se compara con la estimación que hiciste al inicio?
> <details><summary>Ver respuesta</summary>El modelo predice unos $20 560 para F y el valor real es $20 000: el error es de unos $560, menor que $1 000. Para 220 m² predice unos $24 840.</details>

## 5️⃣ Gleihar v0.0: un modelo de lenguaje con los mismos 4 componentes

En el capítulo 6 viste que un modelo de lenguaje estima **P(siguiente token | contexto)**. Gleihar v0.0 es la versión más simple: **ignora el contexto** y solo usa la frecuencia de cada letra.

| Componente | Modelo de terrenos | Gleihar v0.0 |
|---|---|---|
| Datos | 5 terrenos con su precio | Un texto en español |
| Modelo | Recta ŷ = w·x + b | Tabla de probabilidades de cada letra |
| Parámetros | 2 (w y b) | Uno por letra del vocabulario |
| Entrenamiento | Mínimos cuadrados | Contar y dividir |

### ✍️ Tu turno

Completa la función `probabilidades`. Primero la probamos con «banana», el ejemplo resuelto de la teoría.

In [ ]:
def probabilidades(texto):
    """Devuelve un diccionario {símbolo: probabilidad} contando cada símbolo del texto.
    Ejemplo: probabilidades("banana") → {"b": 1/6, "a": 3/6, "n": 2/6}"""
    ...

def revisar_parte5():
    try:
        p = probabilidades("banana")
        ok = abs(p["a"] - 0.5) < 1e-9 and abs(p["n"] - 2/6) < 1e-9 and abs(sum(p.values()) - 1) < 1e-9
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + "probabilidades")
    GLEIHAR["parte5"] = "1/1" if ok else "0/1"
revisar_parte5()

<details><summary><b>🆘 Pista</b></summary>

```python
conteo = {}
for c in texto:
    conteo[c] = conteo.get(c, 0) + 1
total = len(texto)
return {c: n / total for c, n in conteo.items()}
```
</details>

Ahora **entrenamos** Gleihar v0.0 con el inicio del Quijote (1605, dominio público).

> 🔮 **Predice:** ¿cuál será el símbolo más frecuente? (Pista: no es una letra).

In [ ]:
texto = """En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo
de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero,
salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, algún palomino de añadidura los
domingos, consumían las tres partes de su hacienda. El resto della concluían sayo de velarte, calzas de velludo para
las fiestas, con sus pantuflos de lo mesmo, y los días de entresemana se honraba con su vellorí de lo más fino."""
texto = texto.lower().replace("\n", " ")

P = probabilidades(texto)               # ← el entrenamiento: contar y dividir
letras = sorted(P)
print(f"Vocabulario: {len(letras)} símbolos → Gleihar v0.0 tiene {len(letras)} parámetros")
for c, p in sorted(P.items(), key=lambda t: -t[1])[:8]:
    print(f"  {'␣' if c == ' ' else c}  {p:6.2%}")

> ✅ **Comprueba tu predicción:** el símbolo más frecuente es el **espacio**, seguido de las vocales «a», «e» y «o». Por eso el espacio también es un token.

Ahora Gleihar v0.0 **escribe** (inferencia): sortea letras según sus probabilidades. `torch.multinomial` hace el sorteo, como sacar papelitos de una funda. La **semilla** 2026 hace que todos obtengan el mismo resultado.

In [ ]:
generador = torch.Generator().manual_seed(2026)
pesos = torch.tensor([P[c] for c in letras])

def gleihar_v0(longitud=120):
    indices = torch.multinomial(pesos, longitud, replacement=True, generator=generador)
    return "".join(letras[i] for i in indices)

print("Gleihar v0.0 dice:\n")
print(gleihar_v0())

### 🤔 ¿Por qué balbucea?

Porque **no usa contexto**: cada letra se sortea sola. Después de una «q» casi siempre viene una «u», pero Gleihar v0.0 no lo sabe. Ese es el siguiente paso de su camino: en la semana 6 aprenderá a mirar la letra anterior.

> ✍️ **Experimenta:**
> 1. Cambia el texto por un párrafo de una noticia de tu ciudad y vuelve a ejecutar desde la celda del Quijote. ¿Cambian las letras más frecuentes? (Lo necesitarás en la tarea).
> 2. Ejecuta `print(gleihar_v0(300))`. ¿Aparece alguna palabra real por casualidad?

## 6️⃣ Tu reporte

Esta celda resume lo que hiciste. Copia el resultado y pégalo en el cuadro de texto de la **Tarea 1**.

In [ ]:
reporte = dict(GLEIHAR)
reporte["muestra_gleihar_v0"] = gleihar_v0(80)
reporte["fecha"] = time.strftime("%Y-%m-%d %H:%M")
with open("reporte_semana01.json", "w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)
print(json.dumps(reporte, ensure_ascii=False, indent=2))
print("\n✅ Copia este reporte y pégalo en la Tarea 1.")

---

## 🏁 Lo que aprendiste hoy

- ✔️ Construiste un modelo con sus **cuatro componentes**: datos, modelo, pérdida y optimización.
- ✔️ Viste que **aprender = bajar la pérdida** ajustando los parámetros.
- ✔️ Usaste el modelo para **predecir** y lo **evaluaste** con un dato que no había visto.
- ✔️ Construiste **Gleihar v0.0** y reconociste en él los mismos componentes.
- ✔️ Descubriste su límite: le falta **contexto**.

### 🔜 La próxima semana

**Álgebra lineal para IA:** vectores, matrices y producto punto. Son el lenguaje en que se escriben todos los modelos, desde la recta de hoy hasta los transformers.

<div align="center">

**¡Felicitaciones, creaste tus primeros dos modelos de IA! 🎉**

</div>